### ingestar de RAW - BRONZE

In [0]:
dbutils.widgets.removeAll()

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
dbutils.widgets.text("container", "raw")
dbutils.widgets.text("catalogo", "catalog_movie_project")
dbutils.widgets.text("esquema", "bronze")
dbutils.widgets.text("storageName", "adlsmovieproject")

In [0]:
container = dbutils.widgets.get("container")
catalogo = dbutils.widgets.get("catalogo")
## esquema = dbutils.widgets.get("esquema")
storageName = dbutils.widgets.get("storageName")

ruta_m   = f"abfss://{container}@{storageName}.dfs.core.windows.net/1_Movies.csv"
ruta_fd = f"abfss://{container}@{storageName}.dfs.core.windows.net/2_FilmDetails.csv"
ruta_mi = f"abfss://{container}@{storageName}.dfs.core.windows.net/3_MoreInfo.csv"
ruta_pp = f"abfss://{container}@{storageName}.dfs.core.windows.net/4_PosterPath.csv"


In [0]:
#df_circuits = spark.read.option('header', True)\
#                        .option('inferSchema', True)\
#                        .csv(ruta_m)

In [0]:
movie_schema = StructType(fields=[StructField("id", IntegerType(), False),
                                     StructField("title", StringType(), True),
                                     StructField("genres", StringType(), True),
                                     StructField("language", StringType(), True),
                                     StructField("user_score", DoubleType(), True),
                                     StructField("runtime_hour", IntegerType(), True),
                                     StructField("runtime_min", IntegerType(), True),
                                     StructField("release_date", DateType(), True),
                                     StructField("vote_count", IntegerType(), True)
])

In [0]:
film_details_schema = StructType(fields=[StructField("id", IntegerType(), False),
                                     StructField("director", StringType(), True),
                                     StructField("top_billed", StringType(), True),
                                     StructField("budget_usd", DoubleType(), True),
                                     StructField("revenue_usd", DoubleType(), True)
])

In [0]:
moreinfo_schema = StructType(fields=[StructField("id", IntegerType(), False),
                                     StructField("runtime", StringType(), True),
                                     StructField("budget", StringType(), True),
                                     StructField("revenue", StringType(), True),
                                     StructField("film_id", IntegerType(), True)
])

In [0]:
posterpath_schema = StructType(fields=[StructField("id", IntegerType(), False),
                                     StructField("poster_path", StringType(), True),
                                     StructField("backdrop_path", StringType(), True)
])

In [0]:
df_m = spark.read\
.option('header', True)\
.schema(movie_schema)\
.csv(ruta_m)

In [0]:
df_fd = spark.read\
.option('header', True)\
.schema(film_details_schema)\
.csv(ruta_fd)

In [0]:
df_mi = spark.read\
.option('header', True)\
.schema(moreinfo_schema)\
.csv(ruta_mi)

In [0]:
df_pp = spark.read\
.option('header', True)\
.schema(posterpath_schema)\
.csv(ruta_pp)

In [0]:
df_pp_mi = df_mi.alias("mi").join(df_pp.alias("pp"), col("mi.id") == col("pp.id"), "left").drop(col("mi.film_id"),col("pp.id"))

In [0]:
df_pp_mi.write \
  .mode("overwrite") \
  .saveAsTable(f"{catalogo}.bronze.moreinfo_pp")

In [0]:
df_m.write \
  .mode("append") \
  .saveAsTable(f"{catalogo}.bronze.movies")

In [0]:
df_fd.write \
  .mode("append") \
  .saveAsTable(f"{catalogo}.silver.filmdetails_clean")

In [0]:
%sql
select * from catalog_movie_project.bronze.moreinfo_pp

In [0]:
# df_m.display()
# df_fd.display()
# df_mi.display()
#df_pp.display()
df_pp_mi.display()